# 👥 Multi-User Sessions and Persistent Chat Histories

## Learning Objectives
In this notebook, you will learn:
1. **Per-user sessions** - keep isolated histories for many users with one `session_id` per user
2. **Streaming answers per user** - stream a response while the history is still saved
3. **SQLite persistence** - store histories in a database with `SQLChatMessageHistory`
4. **Proof of persistence** - a brand-new chain recalls facts it never saw in memory
5. **File-backed histories** - resume a saved session from a JSON file with `FileChatMessageHistory`

## Where this fits
`03_runnable_with_message_history.ipynb` kept histories in a Python dict, which vanishes when the kernel stops.
Here the same chain serves several users and stores their conversations in SQLite and in files. Only the
`get_session_history` factory changes.

## Prerequisites
- `03_runnable_with_message_history.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

> **⚠️ Legacy API**: `RunnableWithMessageHistory` (deprecated in `langchain-core` 1.3.3) and the chat-message-history classes (1.6.4) are removed in `langchain-core` 2.0. `SQLChatMessageHistory` and `FileChatMessageHistory` come from `langchain-community`, which was sunset in May 2026 (no further maintenance). LangChain now recommends LangGraph persistence instead — a checkpointer
> and a `thread_id` — taught in [`01_memory_and_conversational_agent.ipynb`](06a_memory_and_conversational_agent.ipynb)
> (module 02). This module keeps the older API because existing LCEL code uses it everywhere; the setup cell
> lets its deprecation warnings through so you see them where the deprecated classes are used.


---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library warnings — except LangChain's deprecation warnings, which it lets
through because this notebook uses deprecated APIs on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore")
# This notebook uses deprecated LangChain APIs on purpose: let their deprecation warnings through
warnings.filterwarnings("default", category=LangChainDeprecationWarning)

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: answers every user in this notebook
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 👤 Part 1: One History per User

The `session_id` is the only thing that separates one user's conversation from another's. Any string works:
a user id, a chat id, or both combined.

In [ ]:
# ============================================================================
# PART 1: A CONVERSATION CHAIN SHARED BY ALL USERS
# ============================================================================
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "Act as a helpful AI assistant. Keep answers short."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{human_input}"),
])
chain = prompt | llm | StrOutputParser()

history_store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]


conv_chain = RunnableWithMessageHistory(
    chain, get_session_history, input_messages_key="human_input", history_messages_key="history"
)
print("✅ Multi-user conversation chain ready")

---
## 📡 Part 2: Streaming Answers per User

`.stream()` works through the wrapper too; the full answer is saved to the user's history once the stream ends.
Two users talk to the same chain, then each asks for a summary of *their* conversation.

In [ ]:
# ============================================================================
# PART 2A: A STREAMING HELPER
# ============================================================================
def chat_with_llm(prompt_text: str, session_id: str) -> None:
    """Stream the answer for one user's session."""
    print(f"[{session_id}] User: {prompt_text}\n[{session_id}] AI:   ", end="")
    for chunk in conv_chain.stream({"human_input": prompt_text}, {"configurable": {"session_id": session_id}}):
        print(chunk, end="")
    print("\n")

In [ ]:
# ============================================================================
# PART 2B: TWO USERS, INTERLEAVED
# ============================================================================
chat_with_llm("Hi, I am Sourav. Explain AI in 3 bullet points.", "sourav")
chat_with_llm("Hi, can you explain what an LLM is in 2 bullet points?", "arnab")
chat_with_llm("Now do the same for deep learning.", "sourav")

chat_with_llm("Summarize briefly what we have discussed so far.", "sourav")
chat_with_llm("Summarize briefly what we have discussed so far.", "arnab")

for user, history in history_store.items():
    print(f"{user}: {len(history.messages)} messages stored")

Sourav's summary should mention AI and deep learning; Arnab's only LLMs. Neither sees the other's messages —
isolation comes for free once the store is keyed by `session_id`.

---
## 🗄️ Part 3: Persisting Histories in SQLite

`SQLChatMessageHistory` stores each message as a row in a `message_store` table, keyed by `session_id`.
Pass the database URL with the **`connection=`** keyword: the second positional argument is `table_name`,
so `SQLChatMessageHistory(session_id, "sqlite:///x.db")` silently creates no connection and fails at invoke time.

![SQLite chat history](../assets/img/sqllite.png)

In [ ]:
# ============================================================================
# PART 3: A CHAIN BUILDER BACKED BY SQLITE
# ============================================================================
import os
import sqlite3

from langchain_community.chat_message_histories import SQLChatMessageHistory

DB_FILE = "multi_user_memory.db"
RESET_DB = True  # set to False after a kernel restart to keep the rows written by the last run
if RESET_DB and os.path.exists(DB_FILE):
    os.remove(DB_FILE)  # start this demo from an empty database


def get_sql_history(session_id: str) -> BaseChatMessageHistory:
    return SQLChatMessageHistory(session_id=session_id, connection=f"sqlite:///{DB_FILE}")


def build_chain() -> RunnableWithMessageHistory:
    """A fresh chain object — nothing in memory, everything in the database."""
    sql_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Remember user preferences and facts. Be concise."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])
    return RunnableWithMessageHistory(
        sql_prompt | llm | StrOutputParser(),
        get_sql_history,
        input_messages_key="input",
        history_messages_key="history",
    )


print(f"✅ Histories will be stored in {DB_FILE}")

---
## 🔬 Part 4: Proof That the Database Does the Remembering

Run 1 stores preferences and then deletes the chain object. Between the runs we read the raw rows. Run 2 builds
a brand-new chain with zero in-memory state — if it still recalls the facts, they came from SQLite.

In [ ]:
# ============================================================================
# PART 4A: RUN 1 — STORE PREFERENCES, THEN THROW THE CHAIN AWAY
# ============================================================================
config = {"configurable": {"session_id": "persistent_user"}}

chain_v1 = build_chain()
for message in ["My name is Paulo. I prefer dark mode themes and Python over JavaScript.",
                "I also like my responses concise -- no fluff."]:
    print(f"User: {message}\nAI:   {chain_v1.invoke({'input': message}, config=config)}\n")
del chain_v1

with sqlite3.connect(DB_FILE) as conn:
    rows = conn.execute("SELECT session_id, message FROM message_store ORDER BY id").fetchall()
print(f"Rows in message_store: {len(rows)}")
for session_id, message in rows:
    print(f"  {session_id}: {message[:90]}…")

In [ ]:
# ============================================================================
# PART 4B: RUN 2 — A NEW CHAIN RECALLS FROM THE DATABASE
# ============================================================================
config = {"configurable": {"session_id": "persistent_user"}}  # same session as run 1

chain_v2 = build_chain()
for question in ["What's my name?", "What theme do I prefer?", "How do I like my responses?"]:
    print(f"User: {question}\nAI:   {chain_v2.invoke({'input': question}, config=config)}\n")

with sqlite3.connect(DB_FILE) as conn:
    print(f"Rows after both runs: {conn.execute('SELECT COUNT(*) FROM message_store').fetchone()[0]}")

Run 2 had no Python-side history at all, so every correct recall above was loaded from the 4 rows written in
run 1 (and the row count grew by two per question). The rows also outlive the kernel: after a restart, run the setup and LLM cells, set `RESET_DB = False` in
Part 3, run Part 3, then Part 4B on its own — the answers still come from `multi_user_memory.db`.

---
## 📁 Part 5: File-Backed Histories

`FileChatMessageHistory` stores one session as a JSON file. `chat_history.json` next to this notebook is a
session saved earlier; we copy it and continue the conversation from the copy, so the original stays unchanged.

In [ ]:
# ============================================================================
# PART 5: RESUME A SAVED SESSION FROM A JSON FILE
# ============================================================================
import shutil

from langchain_community.chat_message_histories import FileChatMessageHistory

shutil.copy("chat_history.json", "saved_chat_session.json")
file_history = FileChatMessageHistory("saved_chat_session.json")

print("Loaded from file:")
for message in file_history.messages:
    print(f"  {message.type}: {message.content}")

file_chain = RunnableWithMessageHistory(
    chain, lambda _session_id: file_history, input_messages_key="human_input", history_messages_key="history"
)
reply = file_chain.invoke({"human_input": "What did I say to you first?"}, {"configurable": {"session_id": "file_user"}})
print(f"\nAI: {reply}")
print(f"Messages now in saved_chat_session.json: {len(FileChatMessageHistory('saved_chat_session.json').messages)}")

| Backend | Survives restart | Many users | Typical use |
|---|---|---|---|
| `InMemoryChatMessageHistory` | ❌ | ✅ (one per key) | tests, notebooks |
| `FileChatMessageHistory` | ✅ | one file per session | single-user desktop / CLI tools |
| `SQLChatMessageHistory` | ✅ | ✅ | web apps, services |

For production scale the same interface is implemented by Redis, Postgres and other stores. For an interactive
version of this chatbot, see `chat_stream.py` in this folder (Streamlit, per-user SQLite history).

---
## 📝 Summary

In this notebook, we learned:

### 1. Per-user sessions
- One `session_id` per user isolates histories inside a single chain

### 2. Streaming per user
- `.stream()` works through `RunnableWithMessageHistory`; the full answer is saved afterwards

### 3. SQLite persistence
- `SQLChatMessageHistory(session_id=..., connection="sqlite:///...")` — pass `connection` by keyword

### 4. Proof of persistence
- A new chain with no in-memory state recalled facts from the database rows

### 5. File-backed histories
- `FileChatMessageHistory` saves one session per JSON file and resumes it later

### Next Steps
- **`05_trimming_windowing_and_summary.ipynb`**: keep long conversations within the context window